In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from sklearn.metrics import precision_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input

In [ ]:
random.seed(42)
np.random.seed(42)


n_users = 120
n_items = 400

artists = [f"artist_{i}" for i in range(40)]
genres = ['pop','rock','hiphop','electronic','jazz','classical','folk']

items = []
for i in range(n_items):
  artist = random.choice(artists)
  genre = random.choice(genres)
  year = random.randint(1990,2022)
  title = f"track_{i}"
  items.append({'item_id': i, 'title': title, 'artist': artist, 'genre': genre, 'year': year})

items_df = pd.DataFrame(items)

user_preferences = {}
for u in range(n_users):
    liked_genres = np.random.choice(genres, size=random.randint(1,2), replace=False)
    user_preferences[u] = liked_genres

interactions = []
for u in range(n_users):
    preferred_genres = user_preferences[u]
    n_listens = random.randint(15, 40)
    liked_items = []
    while len(liked_items) < n_listens:
        it = np.random.randint(0, n_items)
        item_genre = items_df.loc[it, 'genre']
        if item_genre in preferred_genres or random.random() < 0.3:
            liked_items.append(it)
    for it in liked_items:
        plays = np.random.poisson(3) + 1
        timestamp = np.random.randint(1_600_000_000, 1_650_000_000)
        interactions.append({'user_id': u, 'item_id': int(it), 'plays': int(plays), 'timestamp': timestamp})

inter_df = pd.DataFrame(interactions)

train_list = []
test_list = []
for u, g in inter_df.groupby('user_id'):
  g_sorted = g.sort_values('timestamp')
  if len(g_sorted) <= 1:
    train_list.append(g_sorted)
    continue
  test_row = g_sorted.iloc[-1]
  train_rows = g_sorted.iloc[:-1]
  test_list.append(test_row.to_frame().T)
  train_list.append(train_rows)


train_df = pd.concat(train_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)


print('Users:', n_users, 'Items:', n_items, 'Train interactions:', len(train_df), 'Test interactions:', len(test_df))

Users: 120 Items: 400 Train interactions: 3318 Test interactions: 120


In [ ]:
def recommend_for_user(user_id, candidate_scores, train_df, k=10):
  seen = set(train_df[train_df.user_id==user_id].item_id.values)
  sorted_items = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
  recs = [i for i, s in sorted_items if i not in seen]
  return recs[:k]

In [ ]:
popularity = train_df.groupby('item_id')['plays'].sum().sort_values(ascending=False)


def popularity_recs(user_id, k=10):
  seen = set(train_df[train_df.user_id==user_id].item_id.values)
  recs = [int(i) for i in popularity.index if int(i) not in seen]
  return recs[:k]

In [ ]:
items_df['text'] = items_df['artist'] + ' ' + items_df['genre'] + ' ' + items_df['title']
vectorizer = TfidfVectorizer(min_df=1)
item_tfidf = vectorizer.fit_transform(items_df['text'])

item_sim = cosine_similarity(item_tfidf, item_tfidf)

def content_based_scores_for_user(user_id):
  user_hist = train_df[train_df.user_id==user_id]
  if user_hist.empty:
    return {int(i): float(popularity.loc[i]) for i in popularity.index}
  weights = user_hist.plays.values
  item_indices = user_hist.item_id.values
  profile = np.average(item_tfidf[item_indices].toarray(), axis=0, weights=weights)
  scores = cosine_similarity(profile.reshape(1,-1), item_tfidf).flatten()
  return {i: float(scores[i]) for i in range(n_items)}

In [ ]:
mat = csr_matrix((train_df['plays'], (train_df['item_id'], train_df['user_id'])), shape=(n_items, n_users))
from sklearn.metrics.pairwise import cosine_similarity as cos
item_item_sim = cos(mat, mat)


def item_based_scores_for_user(user_id):
  user_hist = train_df[train_df.user_id==user_id]
  scores = np.zeros(n_items)
  for _, row in user_hist.iterrows():
    iid = int(row.item_id)
    w = row.plays
    scores += w * item_item_sim[iid]
  return {i: float(scores[i]) for i in range(n_items)}

In [ ]:
user_encoder = LabelEncoder().fit(np.arange(n_users))
item_encoder = LabelEncoder().fit(np.arange(n_items))

train_pairs = train_df[['user_id','item_id','plays']].copy()
expanded = []
for _, r in train_pairs.iterrows():
  times = 1
  for _ in range(times):
    expanded.append((int(r.user_id), int(r.item_id)))
expanded = np.array(expanded)

all_item_ids = np.arange(n_items)
neg_samples = []
for (u,i) in expanded:
  neg = np.random.choice(all_item_ids)
  while ((train_df.user_id==u) & (train_df.item_id==neg)).any():
    neg = np.random.choice(all_item_ids)
  neg_samples.append((u,int(neg)))

pos_users = expanded[:,0]
pos_items = expanded[:,1]
neg_users = np.array([p[0] for p in neg_samples])
neg_items = np.array([p[1] for p in neg_samples])

batch_size = 128
embed_dim = 32

user_input = Input(shape=(), dtype='int32', name='user')
item_input = Input(shape=(), dtype='int32', name='item')

user_emb = layers.Embedding(input_dim=n_users, output_dim=embed_dim, name='user_emb')(user_input)
item_emb = layers.Embedding(input_dim=n_items, output_dim=embed_dim, name='item_emb')(item_input)

user_vec = layers.Flatten()(user_emb)
item_vec = layers.Flatten()(item_emb)

dot = layers.Dot(axes=1)([user_vec, item_vec])

output = layers.Activation('sigmoid')(dot)

model = Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy')

users = np.concatenate([pos_users, neg_users])
items = np.concatenate([pos_items, neg_items])
labels = np.concatenate([np.ones(len(pos_users)), np.zeros(len(neg_users))])

perm = np.random.permutation(len(labels))
users, items, labels = users[perm], items[perm], labels[perm]

model.fit({'user':users,'item':items}, labels, epochs=6, batch_size=batch_size, verbose=1)

user_emb_layer = model.get_layer('user_emb')
item_emb_layer = model.get_layer('item_emb')
user_embeddings = user_emb_layer.get_weights()[0]
item_embeddings = item_emb_layer.get_weights()[0]

Epoch 1/6
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6931
Epoch 2/6
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6916
Epoch 3/6
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6896
Epoch 4/6
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6862
Epoch 5/6
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6807
Epoch 6/6
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6723


In [ ]:
def precision_at_k(recommended, actual, k=10):
  if not actual:
    return 0.0
  rec_k = recommended[:k]
  hits = len(set(rec_k) & set(actual))
  return hits / k


def evaluate(recommender_fn, k=10):
  precisions = []
  for _, row in test_df.iterrows():
    u = int(row.user_id)
    true_item = int(row.item_id)
    scores = recommender_fn(u)
    recs = [i for i, s in sorted(scores.items(), key=lambda x: x[1], reverse=True) if i not in set(train_df[train_df.user_id==u].item_id.values)]
    precisions.append(precision_at_k(recs, [true_item], k))
  return np.mean(precisions)


print('Popularity P@10:', evaluate(lambda u: {i: popularity.loc[i] if i in popularity.index else 0.0 for i in range(n_items)}))
print('Content-based P@10:', evaluate(lambda u: content_based_scores_for_user(u)))
print('Item-based CF P@10:', evaluate(lambda u: item_based_scores_for_user(u)))

Evaluating...
Popularity P@10: 0.004166666666666667
Content-based P@10: 0.006666666666666667
Item-based CF P@10: 0.0016666666666666668


In [ ]:
def show_recommendations(user_id, recs, items_df, k=10):
    print(f"Top-{k} recommendations for user {user_id}:")
    for i, item_id in enumerate(recs[:k]):
        row = items_df[items_df.item_id==item_id].iloc[0]
        print(f"{i+1}. {row['title']} (Artist: {row['artist']}, Genre: {row['genre']})")


In [ ]:
sample_user = 5

pop_recs = popularity_recs(sample_user, 10)
show_recommendations(sample_user, pop_recs, items_df)

content_recs = recommend_for_user(sample_user, content_based_scores_for_user(sample_user), train_df, 10)
show_recommendations(sample_user, content_recs, items_df)

cf_recs = recommend_for_user(sample_user, item_based_scores_for_user(sample_user), train_df, 10)
show_recommendations(sample_user, cf_recs, items_df)



Top-10 recommendations for user 5:
1. track_9 (Artist: artist_27, Genre: hiphop)
2. track_87 (Artist: artist_3, Genre: classical)
3. track_93 (Artist: artist_29, Genre: hiphop)
4. track_148 (Artist: artist_17, Genre: hiphop)
5. track_208 (Artist: artist_8, Genre: jazz)
6. track_120 (Artist: artist_17, Genre: hiphop)
7. track_136 (Artist: artist_39, Genre: classical)
8. track_50 (Artist: artist_32, Genre: jazz)
9. track_115 (Artist: artist_28, Genre: folk)
10. track_376 (Artist: artist_29, Genre: hiphop)
Top-10 recommendations for user 5:
1. track_311 (Artist: artist_17, Genre: rock)
2. track_90 (Artist: artist_30, Genre: rock)
3. track_143 (Artist: artist_30, Genre: rock)
4. track_194 (Artist: artist_8, Genre: rock)
5. track_55 (Artist: artist_19, Genre: rock)
6. track_252 (Artist: artist_5, Genre: rock)
7. track_308 (Artist: artist_5, Genre: rock)
8. track_31 (Artist: artist_4, Genre: rock)
9. track_382 (Artist: artist_6, Genre: rock)
10. track_363 (Artist: artist_18, Genre: rock)
Top

In [ ]:

import math
from itertools import combinations
from collections import defaultdict

K = 10

def get_topk_from_scores(user_id, scores_dict, k=K):
    seen = set(train_df[train_df.user_id==user_id].item_id.values)
    sorted_items = [i for i,s in sorted(scores_dict.items(), key=lambda x: x[1], reverse=True) if i not in seen]
    return sorted_items[:k]

def precision_at_k_list(recs, relevant, k=K):
    if len(recs) == 0: return 0.0
    hits = sum([1 for r in recs[:k] if r in relevant])
    return hits / k

def recall_at_k_list(recs, relevant, k=K):
    if len(relevant) == 0: return 0.0
    hits = sum([1 for r in recs[:k] if r in relevant])
    return hits / len(relevant)

pop_counts = train_df.groupby('item_id')['plays'].sum().reindex(range(n_items)).fillna(0)
pop_rank = pop_counts.rank(ascending=False, method='min')

def evaluate_recommender(recommender_score_fn, k=K):
    precisions, recalls = [], []
    all_recommended = []

    for _, row in test_df.iterrows():
        u = int(row.user_id)
        true_item = int(row.item_id)
        relevant = [true_item]

        scores = recommender_score_fn(u)
        recs = get_topk_from_scores(u, scores, k)

        all_recommended.extend(recs)

        precisions.append(precision_at_k_list(recs, relevant, k))
        recalls.append(recall_at_k_list(recs, relevant, k))

    unique_recs = set(all_recommended)
    coverage = len(unique_recs) / float(n_items)

    diversity_vals = []
    for _, row in test_df.iterrows():
        u = int(row.user_id)
        scores = recommender_score_fn(u)
        recs = get_topk_from_scores(u, scores, k)

        if len(recs) <= 1:
            diversity_vals.append(0.0)
            continue

        pairs = list(combinations(recs, 2))
        diff_count = 0
        for i, j in pairs:
            gi = items_df.loc[items_df.item_id == i, 'genre'].values[0]
            gj = items_df.loc[items_df.item_id == j, 'genre'].values[0]
            if gi != gj:
                diff_count += 1

        diversity_vals.append(diff_count / len(pairs))

    diversity = sum(diversity_vals) / len(diversity_vals)

    ranks = [pop_rank.loc[i] for i in all_recommended]
    max_rank = pop_rank.max()
    novelty = sum([(rank - 1) / (max_rank - 1 + 1e-9) for rank in ranks]) / len(ranks) if ranks else 0.0

    return {
        'P@K': sum(precisions)/len(precisions),
        'Recall@K': sum(recalls)/len(recalls),
        'Coverage': coverage,
        'Diversity': diversity,
        'Novelty': novelty
    }

def popularity_scores_for_user(u):
    return {i: float(pop_counts.loc[i]) for i in range(n_items)}

models = {
    'Popularity': popularity_scores_for_user,
    'Content-based': content_based_scores_for_user,
    'Item-based CF': item_based_scores_for_user
}

results = {}
for name, fn in models.items():
    print('Evaluating', name)
    res = evaluate_recommender(fn, k=K)
    results[name] = res
    print(name)
    for k,v in res.items():
        print(f"  {k}: {v:.4f}")
    print('')

import pandas as pd
results_df = pd.DataFrame(results).T
display(results_df)


Evaluating Popularity
Popularity
  P@K: 0.0042
  Recall@K: 0.0417
  Coverage: 0.0375
  Diversity: 0.7874
  Novelty: 0.0121

Evaluating Content-based
Content-based
  P@K: 0.0067
  Recall@K: 0.0667
  Coverage: 0.9250
  Diversity: 0.3046
  Novelty: 0.4949

Evaluating Item-based CF
Item-based CF
  P@K: 0.0017
  Recall@K: 0.0167
  Coverage: 0.7800
  Diversity: 0.8220
  Novelty: 0.3114



,P@K,Recall@K,Coverage,Diversity,Novelty
Popularity,0.004167,0.041667,0.0375,0.787407,0.012086
Content-based,0.006667,0.066667,0.9250,0.304630,0.494910
Item-based CF,0.001667,0.016667,0.7800,0.822037,0.311353
